# Gold Layer — Brand Analytics

This notebook builds Gold-layer tables from validated Silver data and answers the four Brand Manager questions:
1. Brand#32 total revenue and market share by category
2. Average list-price margin for Brand#32
3. Category dominance and stability over time
4. Products with actual sale prices deviating >15% from list price

In [0]:
%sql
-- Verify Silver tables exist before building Gold
SHOW TABLES IN bda_silver;

SELECT COUNT(*) AS part_count FROM bda_silver.part;
SELECT COUNT(*) AS lineitem_count FROM bda_silver.lineitem;
SELECT COUNT(*) AS orders_count FROM bda_silver.orders;
SELECT COUNT(*) AS partsupp_count FROM bda_silver.partsupp;

## Metric Definitions

| Concept | Definition |
|---|---|
| Revenue (net) | `l_extendedprice × (1 − l_discount)`, excluding tax |
| List price | `p_retailprice` — unit price from the part table |
| Supply cost | `ps_supplycost` for the exact product–supplier pair |
| Unit list-price margin | `p_retailprice − ps_supplycost` |
| Actual unit sale price | `l_extendedprice × (1 − l_discount) / l_quantity` |
| Price deviation | `|actual_unit_price − p_retailprice| / p_retailprice` |
| Reporting date | `o_orderdate` |
| Category | Full `p_type` value (TPC-H has no separate category column) |
| Brand market share | Brand revenue in a category ÷ all revenue in that category |

**Average margin method:** For each product, average margins across all its supplier offers. Then average product-level margins equally across Brand#32 products (each product weighted equally, not sales-weighted).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bda_gold;

CREATE OR REPLACE TABLE bda_gold.brand_sales
USING DELTA AS
SELECT
    l.l_orderkey      AS order_key,
    l.l_linenumber     AS line_number,
    l.l_partkey        AS part_key,
    l.l_suppkey        AS supplier_key,
    o.o_orderdate      AS order_date,
    CAST(date_trunc('MONTH', o.o_orderdate) AS DATE) AS month_start,
    p.p_brand          AS brand,
    p.p_mfgr           AS manufacturer,
    p.p_type           AS category,
    l.l_quantity       AS quantity,
    p.p_retailprice    AS list_unit_price,
    ps.ps_supplycost   AS supply_unit_cost,

    l.l_extendedprice * (1 - l.l_discount)
        AS net_revenue,

    p.p_retailprice - ps.ps_supplycost
        AS list_unit_margin,

    l.l_extendedprice * (1 - l.l_discount) / l.l_quantity
        AS actual_unit_price,

    ABS(
        l.l_extendedprice * (1 - l.l_discount) / l.l_quantity
        - p.p_retailprice
    ) / p.p_retailprice
        AS price_deviation_ratio

FROM bda_silver.lineitem l
JOIN bda_silver.orders   o  ON l.l_orderkey = o.o_orderkey
JOIN bda_silver.part     p  ON l.l_partkey  = p.p_partkey
JOIN bda_silver.partsupp ps ON l.l_partkey  = ps.ps_partkey
                              AND l.l_suppkey = ps.ps_suppkey;

In [0]:
%sql
CREATE OR REPLACE TABLE bda_gold.brand_category_monthly
USING DELTA AS
WITH totals AS (
    SELECT
        month_start,
        category,
        brand,
        SUM(net_revenue) AS brand_revenue
    FROM bda_gold.brand_sales
    GROUP BY month_start, category, brand
)
SELECT
    month_start,
    category,
    brand,
    brand_revenue,
    SUM(brand_revenue) OVER (PARTITION BY month_start, category) AS category_revenue,
    brand_revenue / NULLIF(
        SUM(brand_revenue) OVER (PARTITION BY month_start, category), 0
    ) AS market_share
FROM totals;

CREATE OR REPLACE TABLE bda_gold.part_list_margin
USING DELTA AS
SELECT
    p.p_partkey AS part_key,
    p.p_brand   AS brand,
    p.p_type    AS category,
    COUNT(ps.ps_suppkey) AS supplier_offer_count,
    AVG(p.p_retailprice - ps.ps_supplycost) AS average_list_unit_margin,
    MIN(p.p_retailprice - ps.ps_supplycost) AS minimum_list_unit_margin,
    MAX(p.p_retailprice - ps.ps_supplycost) AS maximum_list_unit_margin
FROM bda_silver.part p
LEFT JOIN bda_silver.partsupp ps ON p.p_partkey = ps.ps_partkey
GROUP BY p.p_partkey, p.p_brand, p.p_type;

CREATE OR REPLACE VIEW bda_gold.price_deviation_products AS
SELECT
    part_key,
    brand,
    category,
    COUNT(*) AS anomalous_line_count,
    MAX(price_deviation_ratio) AS maximum_deviation_ratio
FROM bda_gold.brand_sales
WHERE price_deviation_ratio > 0.15
GROUP BY part_key, brand, category;

In [0]:
%sql
SELECT COUNT(*) AS brand_sales_rows FROM bda_gold.brand_sales;
SELECT COUNT(*) AS monthly_rows FROM bda_gold.brand_category_monthly;
SELECT COUNT(*) AS part_margin_rows FROM bda_gold.part_list_margin;
SELECT COUNT(*) AS products_without_supplier
FROM bda_gold.part_list_margin WHERE supplier_offer_count = 0;
SELECT * FROM bda_gold.brand_sales LIMIT 5;

## Question 1 — Brand#32 Revenue and Category Share

**Metric:** Brand market share = Brand revenue in a category ÷ all revenue in that category.

**Important:** Category totals include ALL brands, not just Brand#32. Filtering to Brand#32 before calculating totals would incorrectly produce 100% share.

In [0]:
%sql
-- Q1a: Brand#32 total revenue
SELECT SUM(net_revenue) AS brand32_total_revenue
FROM bda_gold.brand_sales
WHERE brand = 'Brand#32';

-- Q1b: Brand#32 market share within each category
WITH totals AS (
    SELECT
        category,
        SUM(net_revenue) AS category_revenue,
        SUM(CASE WHEN brand = 'Brand#32' THEN net_revenue ELSE 0 END) AS brand32_revenue
    FROM bda_gold.brand_sales
    GROUP BY category
)
SELECT
    category,
    brand32_revenue,
    category_revenue,
    ROUND(brand32_revenue / NULLIF(category_revenue, 0) * 100, 2) AS brand32_market_share_pct
FROM totals
WHERE brand32_revenue > 0
ORDER BY brand32_revenue DESC;

## Question 2 — Average List-Price Margin for Brand#32

**Metric:** Unit list-price margin = `p_retailprice − ps_supplycost`.

**Method:** For each product, average margins across all its supplier offers. Then average product-level margins equally across all Brand#32 products (equal weight per product, not sales-weighted).

In [0]:
%sql
SELECT
    COUNT(*) AS product_count,
    COUNT(average_list_unit_margin) AS products_with_known_margin,
    ROUND(AVG(average_list_unit_margin), 4) AS average_list_unit_margin
FROM bda_gold.part_list_margin
WHERE brand = 'Brand#32';

## Question 3 — Category Dominance and Stability

**Method:** Calculate all-time market share per brand within each category. Rank brands by share. Then examine the top brand's monthly share stability.

**Stability threshold:** A brand is "stable" if its monthly market share varies by no more than 5 percentage points (max − min).

In [0]:
%sql
-- Q3a: All-time market share and rank by category
WITH totals AS (
    SELECT category, brand, SUM(net_revenue) AS brand_revenue
    FROM bda_gold.brand_sales
    GROUP BY category, brand
),
shares AS (
    SELECT
        category, brand, brand_revenue,
        brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share
    FROM totals
)
SELECT
    category,
    brand,
    ROUND(brand_revenue, 2) AS brand_revenue,
    ROUND(market_share * 100, 2) AS market_share_pct,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY market_share DESC) AS category_rank
FROM shares
ORDER BY market_share DESC, category, brand;

In [0]:
%sql
-- Q3b: Monthly stability of the overall dominant brand in its top category
WITH all_time AS (
    SELECT category, brand,
        brand_revenue / NULLIF(SUM(brand_revenue) OVER (PARTITION BY category), 0) AS market_share
    FROM (
        SELECT category, brand, SUM(net_revenue) AS brand_revenue
        FROM bda_gold.brand_sales
        GROUP BY category, brand
    ) t
),
winner AS (
    SELECT category, brand
    FROM all_time
    ORDER BY market_share DESC
    LIMIT 1
),
monthly AS (
    SELECT m.month_start, m.category, m.brand, m.market_share
    FROM bda_gold.brand_category_monthly m
    JOIN winner w ON m.category = w.category AND m.brand = w.brand
)
SELECT
    brand,
    category,
    COUNT(*) AS months_with_sales,
    ROUND(MIN(market_share) * 100, 2) AS min_share_pct,
    ROUND(MAX(market_share) * 100, 2) AS max_share_pct,
    ROUND((MAX(market_share) - MIN(market_share)) * 100, 2) AS share_range_pp,
    ROUND(STDDEV(market_share) * 100, 2) AS stddev_share_pct
FROM monthly
GROUP BY brand, category;

## Question 4 — Products with Actual Prices Deviating >15% from List Price

**Metric:** Price deviation ratio = `|actual_unit_price − p_retailprice| / p_retailprice`.

A product is "affected" if at least one sale line exceeds the 15% threshold. Actual unit price = `l_extendedprice × (1 − l_discount) / l_quantity` (unit price vs unit price, not line total vs unit price).

In [0]:
%sql
-- Q4a: Total count of anomalous products
SELECT COUNT(*) AS anomalous_product_count
FROM bda_gold.price_deviation_products;

-- Q4b: Concentration by brand and category
SELECT
    brand,
    category,
    COUNT(*) AS anomalous_product_count
FROM bda_gold.price_deviation_products
GROUP BY brand, category
ORDER BY anomalous_product_count DESC;

-- Q4c: Affected product rate (percentage of sold products affected)
WITH affected AS (
    SELECT brand, category, COUNT(DISTINCT part_key) AS affected_products
    FROM bda_gold.brand_sales
    WHERE price_deviation_ratio > 0.15
    GROUP BY brand, category
),
total AS (
    SELECT brand, category, COUNT(DISTINCT part_key) AS total_products
    FROM bda_gold.brand_sales
    GROUP BY brand, category
)
SELECT
    t.brand,
    t.category,
    a.affected_products,
    t.total_products,
    ROUND(a.affected_products / NULLIF(t.total_products, 0) * 100, 2) AS affected_pct
FROM total t
JOIN affected a ON t.brand = a.brand AND t.category = a.category
ORDER BY affected_pct DESC;

## Summary

Gold-layer tables created:
- `gold_layer.brand_sales` — enriched sales fact at line-item grain
- `gold_layer.brand_category_monthly` — monthly brand revenue and market share
- `gold_layer.part_list_margin` — product-level margins from supplier offers
- `gold_layer.price_deviation_products` — view of products exceeding 15% price deviation

All four Brand Manager questions answered. These tables feed `03_validation`, `04_monitoring`, and `sql/brand_share_alert.sql`.